# Prerequisites

## Create Catalog and schema

In [0]:
%sql
use catalog ecommerce;
create schema if not exists bronze;

## Create Initial DFs

In [0]:
%sql
--drop table if exists bronze.orders;


In [0]:
customers_df = spark.read.csv(
    path="/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/initial/customers.csv",
    header=True,
    inferSchema=True,
)

# order_items_df = spark.read.csv(
#     path="/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/initial/order_items.csv",
#     header=True,
#     inferSchema=True,
# )

# orders_df = spark.read.csv(
#     path="/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/initial/orders.csv",
#     header=True,
#     inferSchema=True,
# )

# payments_df = spark.read.csv(
#     path="/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/initial/payments.csv",
#     header=True,
#     inferSchema=True,
# )



In [0]:
customers_df.write.saveAsTable("bronze.customers", mode="APPEND")
# order_items_df.write.saveAsTable("bronze.order_items", mode="APPEND")
# payments_df.write.saveAsTable("bronze.payments", mode="APPEND")

## Orders Table

In [0]:
# from pyspark.sql.types import *
# from pyspark.sql.functions import *

# ORDERS_SCHEMA = StructType([
# StructField("order_id", IntegerType(), True),
# StructField("customer_id", DoubleType(), True),
# StructField("order_date", TimestampType(), True),
# StructField("order_status", StringType(), True),
# StructField("shipping_address", StringType(), True),
# StructField("total_amount", DoubleType(), True),
# StructField("created_at", TimestampType(), True),
# StructField("updated_at", TimestampType(), True)
# ])
 
# input_df = spark.readStream.load(
#     format="CSV", path="/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/initial/orders/", schema=ORDERS_SCHEMA,
#     header=True
# )

# input_df.writeStream.option(
#     "checkpointLocation",
#     "/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/orders/checkpoint",
# ).trigger(availableNow=True).outputMode("APPEND").toTable(
#     "ecommerce.bronze.orders"
# )

In [0]:
# spark.read.table("bronze.orders").limit(5).display()
spark.read.table("bronze.customers").limit(5).display()
# spark.read.table("bronze.order_items").limit(5).display()
# spark.read.table("bronze.payments").limit(5).display()

## Check Dups

In [0]:
%sql
SELECT
    customer_id,
    COUNT(*) AS cnt
FROM ecommerce.bronze.customers
GROUP BY customer_id
HAVING COUNT(*) > 1
limit 3;

## Orders table using AutoLoaders

In [0]:
soruce_configurations = {
    "cloudFiles.format":"csv",
    "header":True,
    "timestampFormat":"yyyy-MM-dd",
    "cloudFiles.schemaLocation":"/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/orders/schema/",
    "cloudFiles.inferColumnTypes":True
}
 
destination_configurations = {
    "checkpointLocation": "/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/orders/checkpoint/",
    "mergeSchema": True,
}
source_df = (
    spark.readStream.format("cloudFiles")
    .options(**soruce_configurations)
    .load("/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/orders/input/")
)

source_df.writeStream.format("DELTA").options(**destination_configurations).outputMode(
    "APPEND"
).trigger(availableNow=True).toTable("ecommerce.bronze.orders")

In [0]:
display(spark.read.table("ecommerce.bronze.orders").count())
spark.read.table("ecommerce.bronze.orders").limit(5).display()

## Order_items table using AutoLoaders

In [0]:
soruce_configurations = {
    "cloudFiles.format":"csv",
    "header":True,
    "timestampFormat":"yyyy-MM-dd",
    "cloudFiles.schemaLocation":"/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/order_items/schema/",
    "cloudFiles.inferColumnTypes":True
}
 
destination_configurations = {
    "checkpointLocation": "/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/order_items/checkpoint/",
    "mergeSchema": True,
}
source_df = (
    spark.readStream.format("cloudFiles")
    .options(**soruce_configurations)
    .load("/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/order_items/input/")
)

source_df.writeStream.format("DELTA").options(**destination_configurations).outputMode(
    "APPEND"
).trigger(availableNow=True).toTable("ecommerce.bronze.order_items")

In [0]:
display(spark.read.table("ecommerce.bronze.order_items").count())
spark.read.table("ecommerce.bronze.order_items").limit(5).display()

## Payments table using AutoLoaders

In [0]:
%sql
--drop table if exists ecommerce.bronze.payments;

In [0]:
soruce_configurations = {
    "cloudFiles.format":"csv",
    "header":True,
    "timestampFormat":"yyyy-MM-dd",
    "cloudFiles.schemaLocation":"/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/payments/schema/",
    "cloudFiles.inferColumnTypes":True
}
 
destination_configurations = {
    "checkpointLocation": "/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/payments/checkpoint/",
    "mergeSchema": True,
}
source_df = (
    spark.readStream.format("cloudFiles")
    .options(**soruce_configurations)
    .load("/Volumes/ecommerce/default/ecommerce_data/ShopSphere_Ecommerce_Capstone_Data/sql_server/payments/input/")
)

source_df.writeStream.format("DELTA").options(**destination_configurations).outputMode(
    "APPEND"
).trigger(availableNow=True).toTable("ecommerce.bronze.payments")

In [0]:
display(spark.read.table("ecommerce.bronze.payments").count())
spark.read.table("ecommerce.bronze.payments").limit(5).display()